# 09 · Resume evidence and key themes

Extract **skills/tools, titles, qualifications and work activities** first. Use activities to support themes such as pipeline engineering; retain the exact evidence behind every result. Dates, certifications and outcomes can follow once role linking is defined.

This is an executable **preliminary design notebook**, extending [08: readiness](08-entity-extraction-readiness.ipynb). It audits local prerequisites and demonstrates manually specified synthetic records. It does **not** perform automatic extraction or establish model accuracy. See the [execution plan and open questions](../plan/2026-09-19-resume-themes/README.md), [original extraction design](../plan/2026-09-13-entity-extraction/README.md), and [data catalog](../docs/data/data-catalog.md).

Only metadata and invented examples are displayed. Each output is capped at five records and 2,000 characters. No downloads, training, annotation or data mutations occur.

## 1. What is useful to extract?

| Stage | Information | Use in the screener | Interpretation boundary |
|---|---|---|---|
| First | Skills and tools | Explain overlap with JD requirements | Mention alone does not establish proficiency |
| First | Job titles, degree and field | Establish role and qualification context | No seniority inferred from tools or school prestige |
| First | Explicit work activities | Support themes, e.g. pipeline engineering | Cite action evidence, not just a tool list |
| Next | Certifications and role-linked dates | Evidence requirements and stated experience | Merge overlapping employment periods before totals |
| Next | Projects, responsibilities, outcomes | Show delivered work and quantified results | Preserve units and role context; do not infer causality |

Leadership requires an explicit activity such as mentoring. Employer/school names are contextual, never prestige scores. Names, contacts, age, gender, nationality and inferred protected traits are excluded from screening features. Theme labels describe documented work, not personality.

In [1]:
import hashlib
import json
import pandas as pd
import pyarrow.parquet as pq
from candidate_screener.config import RAW
from candidate_screener.data.sources import SOURCES
from candidate_screener.annotation.redact import redact_text, assert_clean

print('Ready: metadata inspection and synthetic output-contract checks only.')

Ready: metadata inspection and synthetic output-contract checks only.


## 2. Check local prerequisites

DataTurks (B1) supplies resume entity labels; SkillSpan (B2) supplies JD skill/knowledge spans with a domain shift. ESCO (D1) supports canonical vocabulary; data-jobs (D2) supplies skill frequencies, **not JD text**. Presence below is not provenance verification or repaired-offset readiness. Missing files do not stop the synthetic exercise.

In [2]:
readiness = []
for key in ('dataturks', 'skillspan', 'esco', 'data-jobs'):
    source = SOURCES[key]
    counts = [sum(p.is_file() for p in (RAW / source.directory).glob(a.glob))
              for a in source.artifacts]
    readiness.append({'source': key, 'files': sum(counts),
                      'all_groups_present': all(count >= a.min_files
                                               for count, a in zip(counts, source.artifacts))})
output = pd.DataFrame(readiness).to_string(index=False)
assert len(readiness) <= 5 and len(output) <= 2000
print(output)

   source  files  all_groups_present
dataturks      0               False
skillspan      0               False
     esco      0               False
data-jobs      0               False


A1 is matching data, not entity gold; its fit-label validity failed A13. Djinni (A2) is useful for later descriptive coverage and integration, but lacks entity gold. Preserve its committed train/evaluation partition. C1/C2 additionally need the documented 44 A1 overlaps excluded. The next cell reads parquet footers only: no resume or JD values.

In [3]:
inventory = []
for key in ('fit', 'djinni-cv', 'djinni-jd'):
    for path in sorted((RAW / SOURCES[key].directory).glob('*.parquet')):
        with pq.ParquetFile(path) as parquet:
            inventory.append({'source': key, 'file': path.name,
                              'rows': parquet.metadata.num_rows,
                              'columns': len(parquet.schema_arrow)})
output = (pd.DataFrame(inventory).head(5).to_string(index=False)
          if inventory else 'No local matching parquet files; metadata inspection skipped.')
assert len(output) <= 2000
print(output)

   source                       file   rows  columns
      fit  default-test-0000.parquet   1759        3
      fit default-train-0000.parquet   6241        3
djinni-cv default-train-0000.parquet 210250       11
djinni-jd default-train-0000.parquet 141897       10


## 3. Specify evidence before implementing an extractor

The following documents are invented. Records are **manually specified examples**, not predictions. Coordinates reference an immutable text version and use half-open intervals: `text[start:end] == surface`. A production implementation belongs in `src/candidate_screener/extraction/`, not this notebook.

Redaction runs inside the notebook. Here it must leave the synthetic strings unchanged. Real documents require equal-length masking or an explicit offset map: ordinary redaction can invalidate offsets and does not guarantee removal of every name.

In [4]:
documents = {
    'cv-demo': 'Data Engineer. Built Python and SQL pipelines. Learning Kubernetes. No Docker experience.',
    'jd-demo': 'Python required. Kubernetes preferred. Docker preferred. Java required.'
}
for doc_id, text in documents.items():
    assert redact_text(text) == text, 'Use offset-preserving masking before evidence creation.'
assert_clean(pd.Series(documents), 'synthetic evidence-contract notebook')
text_versions = {doc_id: hashlib.sha256(text.encode()).hexdigest()
                 for doc_id, text in documents.items()}
output = json.dumps(documents, indent=2)
assert len(output) <= 2000
print(output)

{
  "cv-demo": "Data Engineer. Built Python and SQL pipelines. Learning Kubernetes. No Docker experience.",
  "jd-demo": "Python required. Kubernetes preferred. Docker preferred. Java required."
}


In [5]:
# Hand-labelled illustration: index() locates known, unique strings; it is not an extractor.
manual_entities = [
    ('e1', 'Data Engineer', 'JOB_TITLE', 'unknown'),
    ('e2', 'Built Python and SQL pipelines', 'ACTIVITY', 'applied'),
    ('e3', 'Python', 'SKILL', 'applied'),
    ('e4', 'Kubernetes', 'SKILL', 'learning'),
    ('e5', 'Docker', 'SKILL', 'negated'),
]
entities = []
for entity_id, surface, label, status in manual_entities:
    start = documents['cv-demo'].index(surface)
    entities.append({
        'entity_id': entity_id, 'doc_id': 'cv-demo', 'source': 'synthetic',
        'text_version': text_versions['cv-demo'], 'start': start, 'end': start + len(surface),
        'surface': surface, 'label': label, 'canonical_id': None,
        'method': 'manual_demo', 'extractor_version': 'manual-contract-v1',
        'confidence': None, 'evidence_status': status,
    })
output = pd.DataFrame(entities)[['entity_id', 'surface', 'label', 'evidence_status']].to_string(index=False)
assert len(entities) <= 5 and len(output) <= 2000
print(output)

entity_id                        surface     label evidence_status
       e1                  Data Engineer JOB_TITLE         unknown
       e2 Built Python and SQL pipelines  ACTIVITY         applied
       e3                         Python     SKILL         applied
       e4                     Kubernetes     SKILL        learning
       e5                         Docker     SKILL         negated


The example includes only selected spans; SQL and other context would need additional records in a complete annotation. Null canonical IDs mean mapping is unresolved. Null confidence avoids inventing probabilities. `unknown` means the evidence does not specify applied/learning/negated status; title evidence alone cannot assign proficiency.

In [6]:
entity_by_id = {row['entity_id']: row for row in entities}
relations = [{'source_id': 'e3', 'relation': 'used_in', 'target_id': 'e2'}]
themes = [{
    'theme_id': 'demo:pipeline-engineering', 'label': 'Pipeline engineering',
    'taxonomy_version': 'illustrative-v1', 'supporting_entity_ids': ['e2'],
    'method': 'manual_demo',
}]
# A theme is justified by a stated activity; learning Kubernetes is not cloud-operations evidence.
for theme in themes:
    support = [entity_by_id[eid] for eid in theme['supporting_entity_ids']]
    assert support and all(e['label'] == 'ACTIVITY' and e['evidence_status'] == 'applied'
                           for e in support)
output = json.dumps(themes, indent=2)
assert len(themes) <= 5 and len(output) <= 2000
print(output)

[
  {
    "theme_id": "demo:pipeline-engineering",
    "label": "Pipeline engineering",
    "taxonomy_version": "illustrative-v1",
    "supporting_entity_ids": [
      "e2"
    ],
    "method": "manual_demo"
  }
]


## 4. Compare with JD evidence without treating mentions as qualification

Required/preferred status comes from explicit JD wording, not an occupation taxonomy. The following links and statuses are manually specified too. Learning, negation and missing evidence are different states; a missing mention means **not evidenced**, not absent ability. These are screening questions, not automatic rejection rules.

In [7]:
manual_requirements = [
    ('Python required', 'required', 'e3', 'applied'),
    ('Kubernetes preferred', 'preferred', 'e4', 'learning'),
    ('Docker preferred', 'preferred', 'e5', 'negated'),
    ('Java required', 'required', None, 'not_evidenced'),
]
requirements = []
for surface, priority, evidence_id, status in manual_requirements:
    start = documents['jd-demo'].index(surface)
    requirements.append({'doc_id': 'jd-demo', 'text_version': text_versions['jd-demo'],
                         'start': start, 'end': start + len(surface), 'surface': surface,
                         'priority': priority, 'evidence_id': evidence_id, 'status': status,
                         'method': 'manual_demo'})
output = pd.DataFrame(requirements)[['surface', 'priority', 'evidence_id', 'status']].to_string(index=False)
assert len(requirements) <= 5 and len(output) <= 2000
print(output)

             surface  priority evidence_id        status
     Python required  required          e3       applied
Kubernetes preferred preferred          e4      learning
    Docker preferred preferred          e5       negated
       Java required  required         NaN not_evidenced


In [8]:
assert len(entity_by_id) == len(entities)
for row in entities + requirements:
    text = documents[row['doc_id']]
    assert row['text_version'] == hashlib.sha256(text.encode()).hexdigest()
    assert 0 <= row['start'] < row['end'] <= len(text)
    assert text[row['start']:row['end']] == row['surface']
for relation in relations:
    assert relation['source_id'] in entity_by_id and relation['target_id'] in entity_by_id
for requirement in requirements:
    evidence_id = requirement['evidence_id']
    if evidence_id is None:
        assert requirement['status'] == 'not_evidenced'
    else:
        assert requirement['status'] == entity_by_id[evidence_id]['evidence_status']
assert entity_by_id['e4']['evidence_status'] != 'applied'
assert entity_by_id['e5']['evidence_status'] != 'applied'
print('Synthetic contract checks passed: offsets, text versions, links and evidence distinctions.')

Synthetic contract checks passed: offsets, text versions, links and evidence distinctions.


These assertions validate consistency of the illustration, **not extraction accuracy**. No gold dataset or performance estimate is produced. Theme inference, context classification, relation linking and requirement parsing remain to be implemented and evaluated independently.

## 5. Execution steps and completion gates

1. **Restore and verify prerequisites.** Use registered acquisition instructions for B1/B2 and D1/D2. Run `uv run python -m candidate_screener.data.verify --all`. Gate: provenance verified and source-use scope agreed.
2. **Freeze supervision and text coordinates.** Repair B1 inclusive offsets; quarantine ambiguous repairs. Original Skills blobs remain gold; automatic segmentation is silver. Deduplicate/group before B1 document-level cross-validation; preserve B2 published splits and independent BIO layers. Protect A2's committed partition. Gate: stable text versions, offset checks and leakage audit.
3. **Build package schema and phrase/rule baseline.** Add source adapters and versioned aliases. Cover C++, C#, .NET, ambiguous Go/R, learning and negation. Gate: meaningful regression invariants demonstrated to fail against guarded defects; no model construction in notebooks.
4. **Evaluate extraction and mapping separately.** Report exact span+label precision/recall/F1, class support and B1 CV mean/std; B2 per split and layer. A2 coverage is descriptive, not accuracy. Freeze rules before held-out evaluation. Gate: reproducible baseline records and checks.
5. **Compare BERT under the same protocol.** Resolve budget and useful-gain threshold; document subword alignment and long-document windows. Gate: measured extraction gain and latency trade-off.
6. **Add activity-backed themes and role/date relations.** Merge overlapping employment periods before experience totals. Version the taxonomy and retain support IDs. Gate: suitable evaluation labels for unsupported claims and theme precision/recall. Any additional human annotation requires revisiting the one-session budget.
7. **Integrate evidence with retrieve-then-rank.** Evaluate ranking benefit separately using Recall@10 and separate generic/targeted figures. Precision@k retains its soft judged-supply ceiling; A1 fit labels and deferred option D labels are not extraction gold. Gate: measured ranking gain and evidence that can be inspected in the UI.

Run repository checks with `uv run pytest` and applicable committed-record checks. See the [plan](../plan/2026-09-19-resume-themes/README.md) for validation outcomes. Open decisions EE1–EE5 cover production scope, restoration, BERT budget, theme evaluation and role/date policy; this notebook does not resolve those choices.